# AgentCoder Iteration Analysis

This notebook analyzes the code generated and test cases produced by the AgentCoder system for HumanEval problems across multiple iterations. We'll examine how the agent improves solutions through iterative refinement.

In [1]:
# Import necessary libraries
import json
import gzip
from pathlib import Path
from collections import defaultdict
from human_eval.data import write_jsonl, read_problems

# Setup paths and configuration
project_root = Path(__doc__).resolve().parent.parent.parent.parent
subset_path = project_root / "data/human_eval/HumanEval_20.jsonl.gz"
generations_path = project_root / "data/human_eval/generations/agent_coder"

# Load HumanEval problems
human_eval_problems = read_problems(str(subset_path))
print(f"Loaded {len(human_eval_problems)} HumanEval problems")

Loaded 20 HumanEval problems


In [2]:
# Configuration
CONFIG = {
    "llm_model": "qwen2.5-coder_7b",
    "max_iterations": 3,
    "num_samples_per_task": 1,
    "subset_size": 20
}

def read_jsonl_file(file_path):
    """Read a JSONL file and return list of dictionaries."""
    data = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line.strip()))
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except json.JSONDecodeError as e:
        print(f"JSON decode error in {file_path}: {e}")
    return data

def load_iteration_data():
    """Load iteration data and results for all iterations."""
    file_template = "{llm_model}-subset{subset_size}-max{max_iterations}-numSamples{num_samples_per_task}-iter{iteration}.jsonl"
    
    iteration_data = {}
    results_per_iteration = {}
    
    print(f"Loading data for model: {CONFIG['llm_model']}")
    print(f"Max iterations: {CONFIG['max_iterations']}")
    print(f"Samples per task: {CONFIG['num_samples_per_task']}")
    print()
    
    for iteration in range(1, CONFIG['max_iterations'] + 1):
        # Load generation data
        filename = file_template.format(iteration=iteration, **CONFIG)
        file_path = generations_path / filename
        
        print(f"Loading iteration {iteration} from: {filename}")
        generation_data = read_jsonl_file(file_path)
        iteration_data[iteration] = generation_data
        
        # Load results data
        results_filename = filename + "_results.jsonl"
        results_path = generations_path / results_filename
        
        print(f"Loading results from: {results_filename}")
        results_data = read_jsonl_file(results_path)
        results_per_iteration[iteration] = results_data
        
        print(f"  Generation records: {len(generation_data)}")
        print(f"  Results records: {len(results_data)}")
        print()
    
    return iteration_data, results_per_iteration

# Load all iteration data
iteration_data, results_per_iteration = load_iteration_data()

Loading data for model: qwen2.5-coder_7b
Max iterations: 3
Samples per task: 1

Loading iteration 1 from: qwen2.5-coder_7b-subset20-max3-numSamples1-iter1.jsonl
Loading results from: qwen2.5-coder_7b-subset20-max3-numSamples1-iter1.jsonl_results.jsonl
  Generation records: 20
  Results records: 20

Loading iteration 2 from: qwen2.5-coder_7b-subset20-max3-numSamples1-iter2.jsonl
Loading results from: qwen2.5-coder_7b-subset20-max3-numSamples1-iter2.jsonl_results.jsonl
  Generation records: 20
  Results records: 20

Loading iteration 3 from: qwen2.5-coder_7b-subset20-max3-numSamples1-iter3.jsonl
Loading results from: qwen2.5-coder_7b-subset20-max3-numSamples1-iter3.jsonl_results.jsonl
  Generation records: 20
  Results records: 20



In [32]:
from common import CodeProcessor, create_safe_test_environment
code_processor = CodeProcessor()
sandbox = create_safe_test_environment()

# check iteration 2 and iteration 3 differences
current_iteration = 3
previous_iteration = current_iteration - 1
for i, generation  in enumerate(iteration_data[current_iteration]):
    if generation['iteration'] != 3:
        continue

    
    task_id = generation['task_id']
    test_cases = generation['test_cases']
    refined_completion = generation['completion']
    previous_completion = iteration_data[previous_iteration][i]['completion']
    current_test_results_summary = generation['test_results_summary']
    previous_test_results_summary = iteration_data[previous_iteration][i]['test_results_summary']
    problem = human_eval_problems[task_id]

    print("=" * 80)
    print(f"Task ID: {task_id}")


    test_script = code_processor.build_test_script(problem['prompt'] + "\n" + refined_completion, test_cases)
    print(test_script)
    test_results = sandbox.execute_test_script(test_script)
    print("Failed Test Results:")
    print("From Log File: ", current_test_results_summary)

    print("Rerun:", test_results.test_summary)
    print(test_results.test_details.failed_test_details)

    print("Test Errors: ")
    print(test_results.test_details.error_test_details)
    print("=" * 80)

Task ID: HumanEval/129
import unittest

# Programmer Code
def minPath(grid, k):
    """
    Given a grid with N rows and N columns (N >= 2) and a positive integer k, 
    each cell of the grid contains a value. Every integer in the range [1, N * N]
    inclusive appears exactly once on the cells of the grid.

    You have to find the minimum path of length k in the grid. You can start
    from any cell, and in each step you can move to any of the neighbor cells,
    in other words, you can go to cells which share an edge with you current
    cell.
    Please note that a path of length k means visiting exactly k cells (not
    necessarily distinct).
    You CANNOT go off the grid.
    A path A (of length k) is considered less than a path B (of length k) if
    after making the ordered lists of the values on the cells that A and B go
    through (let's call them lst_A and lst_B), lst_A is lexicographically less
    than lst_B, in other words, there exist an integer index i (1 <= i <= k)


In [ ]:
import unittest

# Programmer Code
def strange_sort_list(lst):
    '''
    Given list of integers, return list in strange order.
    Strange sorting, is when you start with the minimum value,
    then maximum of the remaining integers, then minimum and so on.

    Examples:
    strange_sort_list([1, 2, 3, 4]) == [1, 4, 2, 3]
    strange_sort_list([5, 5, 5, 5]) == [5, 5, 5, 5]
    strange_sort_list([]) == []
    '''


    result = []

    while lst:
        if lst:
            min_val = min(lst)
            result.append(min_val)

        if lst:
            max_val = max(lst)
            result.append(max_val)

        if len(lst) == 1:
            result.append(lst.pop())

    return result

# Tester Code
class TestStrangeSortList(unittest.TestCase):

    # Basic Test Cases
    
    def test_basic_sorting(self):
        """
        This test checks if the function correctly sorts a list of integers in the strange order.
        """
        self.assertEqual(strange_sort_list([1, 2, 3, 4]), [1, 4, 2, 3])
    
    def test_identical_elements(self):
        """
        This test verifies that the function handles a list with identical elements correctly.
        """
        self.assertEqual(strange_sort_list([5, 5, 5, 5]), [5, 5, 5, 5])

    def test_empty_list(self):
        """
        This test checks if the function can handle an empty list properly.
        """
        self.assertEqual(strange_sort_list([]), [])

    # Edge Test Cases
    
    def test_single_element(self):
        """
        This test ensures that a single-element list is returned unchanged.
        """
        self.assertEqual(strange_sort_list([42]), [42])

    def test_reversed_order(self):
        """
        This test verifies if the function correctly sorts a list in reverse order.
        """
        self.assertEqual(strange_sort_list([9, 8, 7, 6]), [6, 9, 7, 8])

    def test_negative_numbers(self):
        """
        This test checks that the function can handle lists with negative numbers correctly.
        """
        self.assertEqual(strange_sort_list([-1, -2, -3, -4]), [-4, -1, -3, -2])
    
    # Large Scale Test Cases
    
    def test_large_random_numbers(self):
        """
        This test evaluates the function's performance on a list with large random numbers.
        """
        import random
        test_list = [random.randint(0, 1000) for _ in range(1000)]
        sorted_list = strange_sort_list(test_list)
        self.assertTrue(sorted_list == sorted(test_list, key=lambda x: (x % 2, x)))
    
    def test_large_identical_numbers(self):
        """
        This test checks if the function performs well with a large list of identical numbers.
        """
        import random
        test_list = [random.randint(0, 1) for _ in range(10000)]
        sorted_list = strange_sort_list(test_list)
        self.assertEqual(sorted_list, [0] * (len(test_list) // 2) + [1] * (len(test_list) // 2))


unittest.main(argv=[''], verbosity=2, exit=False)

test_basic_functionality (__main__.TestCountNums.test_basic_functionality)
Basic functionality: Check if the function returns correct counts for simple arrays. ... FAIL
test_edge_cases (__main__.TestCountNums.test_edge_cases)
Edge cases: Check if the function handles edge scenarios correctly. ... FAIL
test_large_scale_performance (__main__.TestCountNums.test_large_scale_performance)
Large scale performance: Check if the function performs well with large data samples. ... FAIL
test_basic_full_single_digit_span (__main__.TestGenerateIntegers.test_basic_full_single_digit_span) ... ok
test_basic_inclusive_bounds (__main__.TestGenerateIntegers.test_basic_inclusive_bounds) ... ok
test_basic_partial_overlap (__main__.TestGenerateIntegers.test_basic_partial_overlap) ... ok
test_basic_reversed_inputs (__main__.TestGenerateIntegers.test_basic_reversed_inputs) ... ok
test_basic_single_point_bounds (__main__.TestGenerateIntegers.test_basic_single_point_bounds) ... ok
test_edge_both_bounds_below_tw